# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the `mlcroissant` library and referencing dataset entities by their `@id` fields.

### Dataset Source
This dataset is described by a Croissant schema:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as a Python object, not a dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their fields (all referenced by `@id`).

In [ ]:
# List available record sets (@id, name) and their fields' @id
print("Available record sets:")
record_sets_info = []
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id}")
    print(f"  name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}")
    record_sets_info.append({
        'id': rs.id,
        'name': rs.name,
        'field_ids': [f.id for f in rs.fields]
    })

# Also print columns for each field if available (by @id)
print("\nField columns by @id where defined:")
for rs in dataset.metadata.record_sets:
    for field in rs.fields:
        if hasattr(field, "columns") and field.columns:
            print(f"Field @id: {field.id}")
            for col in field.columns:
                print(f"  - column @id: {col.id}, name: {col.name}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All record sets, fields, and columns are referenced by their `@id`.

In [ ]:
# Extract data from each record set by its @id
from collections import OrderedDict

dataframes = {}
record_set_ids = [rs['id'] for rs in record_sets_info]
print(f"Record set @id(s): {record_set_ids}\n")

# Display how many records for each record set
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"  No records found for this record set.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {len(df)} records. Columns (@id): {list(df.columns)}\n")

# For demonstration, pick the first available record set
main_rs_id = record_set_ids[0]
print(f"Using main RecordSet @id: {main_rs_id}")
df = dataframes[main_rs_id]
print("First five records:")
display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing—such as filtering, normalizing a numeric field, and grouping—using only `@id`-referenced fields.

In [ ]:
# Identify numeric fields by inspecting field @ids and column names
print("Columns in DataFrame (by @id):")
print(list(df.columns))

# Common numeric field candidates (based on dataset description):
# E.g., Age, Diagnosis Interval, some lab/pathological measurements.
# Let's try 'age' or similar if present

# Try to find a likely numeric column by @id:
numeric_field_candidates = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'count' in col.lower() or 'n_' in col.lower()]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    numeric_field = df.select_dtypes(include=['number']).columns[0] if not df.select_dtypes(include=['number']).empty else df.columns[0]
    print(f"Fallback numeric field @id: {numeric_field}")

# Remove outlier rows if necessary (e.g., values > 3 std above mean)
field_series = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = field_series.mean() + field_series.std() * 3
filtered_df = df[field_series < threshold].copy()
filtered_df = filtered_df[field_series > 0]  # Only positive, valid entries
print(f"Filtered records with {numeric_field} > 0 and < {threshold:.2f}:")
display(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - field_series.mean()) / field_series.std()
print(f"Normalized {numeric_field} (mean-zero, std-one):")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if present (e.g., sex, anatomical site)
group_candidates = [col for col in df.columns if 'sex' in col.lower() or 'site' in col.lower() or 'group' in col.lower() or 'category' in col.lower() or 'stage' in col.lower()]
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_' + numeric_field)
    print(f"Grouped (mean) of {numeric_field} by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize numeric distributions and groupwise differences in the dataset (all variables by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), bins=15, kde=True)
plt.title(f'Distribution of {numeric_field} (@id)')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouped_df created:
if 'grouped_df' in locals():
    grouped_df = grouped_df.reset_index()
    plt.figure(figsize=(8,5))
    sns.barplot(data=grouped_df, x=group_field, y='mean_' + numeric_field)
    plt.title(f'Mean {numeric_field} by {group_field} (@id)')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² colorectal cancer dataset described by a Croissant schema and explored its structure using `mlcroissant`.

- **All access and operations referenced only `@id`s** for record sets, fields, and columns.
- We previewed available record sets, loaded data into DataFrames, filtered and normalized a numeric field using only `@id`s, and compared values by group when possible.
- The dataset enables analysis of clinicopathological properties of secondary colorectal cancer in cancer survivors, including anatomical, demographic, and molecular factors.

For further analyses or machine learning workflows, follow the same approach of referencing data fields by `@id` as demonstrated above.